Insper

# Aula 09 - Spark ML - Machine Learning com Spark

Vamos fazer o setup de nosso ambiente Spark.

In [ ]:
# Criar a sessao do Spark
from pyspark.sql import SparkSession
spark = SparkSession \
            .builder \
            .master("local[*]") \
            .appName("MUDE_AQUI") \   # ALTEREM AQUI!!!
            .getOrCreate()

## Modelo de regressão linear

Vamos usar o clássico conjunto de dados de Advertising para predizer o número de vendas de um produto, em função dos valores gastos em campanhas de TV, Radio e Jornal.

Iniciamos com a leitura do conjunto de dados.

In [ ]:
data = spark.read.csv('../10_dados/sparkML/Advertising.csv',
                      header=True,
                      inferSchema=True)

Exiba as primeiras cinco linhas do dataset.

Remova a coluna `_c0`.

Exiba novamente as primeiras cinco linhas.

Vamos agora exibir as principais estatísticas descritivas do conjunto de dados.

O Spark faz uso de uma interface similar ao Scikit-Learn para desenvolver modelos preditivos. Baseado no conceito de `transformer`, nós vamos transformando o dataset em outro dataset com os ajustes necessários para o desenvolvimento de nosso modelo.

A estrutura de dados mais utilizada para o desenvolvimento de modelos é o `Vector` que possui, dentre outras funções, um bom suporte para dados esparsos.

Agora vamos importar o `VectorAssembler` que está em `pyspark.ml.feature` e também o modelo de regrssão linear (`LinearRegression`) que está em `pyspark.ml.regression`.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

Vamos dividir os dados no conjunto de treino (70%) e teste (30%), como segue:

Para desenvolver seu modelo em Spark, você precisará ter uma coluna denominada `features`, que será resultado de todos os processos de transformação de dados necessários para o correto desenvolvimento dos modelos.

Vamos criar uma variável chamada `vec` que será o nosso `VectorAssembler`. Devemos informar quais colunas serão concatenadas em um `vector` e qual será o nome desse `vector`.

Feito isso, podemos ver o resultado usando a função `transform()`.

Exiba as cinco primeiras linhas de train e, posteriormente, de teste.

É possível observar a coluna `features`. Veja o schema do dataframe.

Agora chegou a vez de criarmos o nosso modelo de regressão linear. Você precisa informar dois parâmetros: `featureCol` (qual é a coluna que possui um vector das features) e `labelCol` que representa a coluna que possui a variável dependente.

Análogo ao `scikit-learn`, use o método `fit()`

O atributo `coefficients` e o atributo `intercept` apresentam, respectivamente, os coeficientes da regressão linear e o valor do intercepto.

E podemos usar a função `evaluate` para criar uma variável que nos permitirá obter as métricas comuns de desempenho do modelo.

# Profissionalizando nossos modelos com Pipeline


De fato, há inúmeros procedimentos que fazemos com os dados antes de treinar um modelo. Há processos de limpeza, padronização, one hot encoding, etc.

Veja nesse link [https://spark.apache.org/docs/latest/ml-features.html](https://spark.apache.org/docs/latest/ml-features.html) as mais diversa funções que podemos fazer nos dados com o Spark. Há códigos de exemplo para lhe auxiliar no aprendizado.

Vamos comecar importando `Pipeline` que está em `pyspark.ml`. E com isso vamos definir um pipeline formado por 2 estágios: vec (transforma as features em vector) e lr (nosso modelo de regressão linear)

In [ ]:
from pyspark.ml import Pipeline

E agora criamos nosso pipeline

Vamos dividir novamente os dados em treino e teste

Agora podemos usar a função `fit()` do pipeline

Execute a função `transform` sob o conjunto de teste (`test`) e salve em `pred`.

## Melhorando nosso pipeline com feature engineering

Lembre-se desse link para ver as mais usadas transformações nos dados: [https://spark.apache.org/docs/latest/ml-features.html](https://spark.apache.org/docs/latest/ml-features.html)


Vamos desenvolver outro modelo preditivo. Agora para predizer o gasto em planos de saúde. Teremos variáveis categóricas e contínuas.

Para as variáveis categóricas, vamos ver como aplicar `OneHotEncoding`.

In [ ]:
gasto = spark.read.csv('../10_dados/sparkML/gasto.csv', header=True, inferSchema=True)

Exiba as cinco primeiras linhas

Exiba o schema.

Separe agora em treino (70%) e teste (30%)

Vamos agora definir as variáveis que são categóricas e quais são numéricas

Para fazer OneHotEncoder no Spark, primeiro precisamos transformar valores (`yes/no`) em números (`0/1`). Para isso temos que usar duas funções `StringIndexer` (que irá converter rótulos em valores) e posteriormente a `OneHotEncoder`. O resultado da `StringIndexer` é um vetor esparso. Você sabe dizer a utilidade disso?


Exemplo de um vetor esparso:

```
DenseVector(0, 0, 0, 7, 0, 2, 0, 0, 0, 0)
SparseVector(10, [3, 5], [7, 2])
```

In [ ]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer

Como sempre temos colunas como `input` e novas colunas como `output`, vamos definir o nome das colunas de output para cada uma das funções

Agora vamos criar nosso StringIndexer.

**Pergunta**: o que fazemos quando indexamos no treino e no teste há um valor desconhecido?

E agora criamos nosso `OneHotEncoder`.

**Pergunta**: qual deve ser o input do OneHotEncoder?

E também precisamos definir todas as colunas que serão usadas no vector que representará as features.

Criamos o `VectorAssembler`

Vamos ver se você entendeu: Obtenha o resultado do oneEncoder para o conjunto de treino. Selecione apenas as 20 primeiras linhas, com as colunas `region`, `regionIndex` e `regionOHE` para verificar o resultado.

Criamos agora nosso modelo de regressão linear

Criamos também nosso pipeline, composto pelos estágios:
- stringIndexer
- oheEncoder
- vecAssembler
- lr

Aplicamos fit e, posteriormente, transform

Como avaliar agora um modelo que é um pipeline. para esse caso, temos que construir um objeto da classe `RegressionEvaluator`.

# Salvando o modelo

É uma boa prática salvar seu modelo para utilizar em outros momentos.

# Carregando o modelo

In [ ]:
from pyspark.ml import PipelineModel

# Otimização de hiperparâmetros e Validação Cruzada

Vamos construir um modelo para nosso problema em questão agora fazendo uso de um `RandomForestRegressor`. Vamos querer otimizar 2 hiperparâmetros (`maxDepth` e `numTrees`), e obter seus valores a partir de um processo de validação cruzada com 5-folds.

In [ ]:
from pyspark.ml.regression import RandomForestRegressor

Para executar o grid, precisamos criar um objeto da classe `ParamGridBuilder`

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder

E criamos também um `RegressionEvaluator`

Para executar a validação cruzada, nós precisamos importar a função `CrossValidator` de `pyspark.ml.tuning`.

In [ ]:
from pyspark.ml.tuning import CrossValidator

E podemos obter nosso modelo a partir do `fit`.

E também podemos verificar todas as trials feitas no processo de otimização.

# Link para referência adicional

Vejam aqui um exemplo de um pipeline de **classificação**.

https://swan-gallery.web.cern.ch/notebooks/SparkTraining/notebooks/ML_Demo1_Classifier.html